# Colab Remote Inference Setup (Qwen 3.5 + LoRA)

Run this notebook on a Google Colab **GPU** runtime.
It:
1. Checks the GPU
2. Clones your project (or copies from Drive)
3. Installs dependencies
4. **Downloads the base model** with live progress
5. **Loads your fine-tuned LoRA weights** from Google Drive
6. Starts the API server in a background thread (output visible here)
7. Opens an ngrok tunnel and prints the public URL

## 1) Runtime Check
Make sure you have set the Colab runtime to **GPU** before running.

In [ ]:
import torch

cuda_ok = torch.cuda.is_available()
print('CUDA available :', cuda_ok)
if cuda_ok:
    print('GPU            :', torch.cuda.get_device_name(0))
    print('VRAM (GB)      :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print('⚠️  No GPU detected — go to Runtime → Change runtime type → GPU')

## 2) Get Project Files
Choose **one** option. The other can stay commented out.

In [ ]:
# Option A: Clone from GitHub
import os

if not os.path.isdir('/content/LLM_Project'):
    !git clone https://github.com/musfirbaig/LLM_Project.git /content/LLM_Project
else:
    print('Repo already present — pulling latest changes')
    !git -C /content/LLM_Project pull

os.chdir('/content/LLM_Project')
print('Working directory:', os.getcwd())

In [ ]:
# Option B: Copy from Google Drive (uncomment if you prefer Drive over GitHub)
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil, os
# if not os.path.isdir('/content/LLM_Project'):
#     shutil.copytree('/content/drive/MyDrive/LLM_Project', '/content/LLM_Project')
# os.chdir('/content/LLM_Project')
# print('Working directory:', os.getcwd())

## 3) Install Dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q "bitsandbytes>=0.43.1" pyngrok
print('✅ Dependencies installed')

## 4) Download Base Model with Progress

This cell downloads `Qwen/Qwen3.5-4B` from Hugging Face and shows a **live download progress bar**.
The files are cached in `/root/.cache/huggingface` so subsequent runs are instant.

> If you need to use a Hugging Face token (gated model) set `HF_TOKEN` in the cell below.

In [ ]:
import os

# ── Optional: set your HF token if the model is gated ──────────────────────
# os.environ['HF_TOKEN'] = 'hf_YOUR_TOKEN_HERE'

BASE_MODEL_ID = os.environ.get('NUST_BANK_BASE_MODEL', 'Qwen/Qwen3.5-4B')
print(f'Base model : {BASE_MODEL_ID}')
print('Downloading model weights (cached on repeat runs) ...')

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Determine dtype / device
if torch.cuda.is_available():
    try:
        from transformers import BitsAndBytesConfig
        quant_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.float16,
        )
        dtype = torch.float16
        device_map = 'auto'
        device_label = f'GPU ({torch.cuda.get_device_name(0)}) · 4-bit quantised'
    except ImportError:
        quant_cfg = None
        dtype = torch.float16
        device_map = 'auto'
        device_label = f'GPU ({torch.cuda.get_device_name(0)})'
else:
    quant_cfg = None
    dtype = torch.bfloat16
    device_map = 'cpu'
    device_label = 'CPU · bfloat16'

print(f'Device     : {device_label}')

import time
t0 = time.time()

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=dtype,
    device_map=device_map,
    quantization_config=quant_cfg,
    trust_remote_code=True,
)

print(f'\n✅ Base model downloaded & loaded in {time.time() - t0:.1f}s')
print(f'   Parameters : {sum(p.numel() for p in base_model.parameters()) / 1e9:.2f}B')

## 5) Load Fine-Tuned LoRA Weights

Mount Google Drive and point `ADAPTER_PATH` to where your `qwen3.5_banking_lora` folder lives.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os

# ── Edit this path to match where your adapter folder is on Drive ──────────
ADAPTER_PATH = '/content/drive/MyDrive/qwen3.5_banking_lora'

# Alternatively, if the adapter is inside the cloned repo:
# ADAPTER_PATH = '/content/LLM_Project/qwen3.5_banking_lora'

if not os.path.isdir(ADAPTER_PATH):
    raise FileNotFoundError(
        f'Adapter not found at {ADAPTER_PATH}.\n'
        'Please update ADAPTER_PATH to the correct Drive location.'
    )

print(f'Adapter found at : {ADAPTER_PATH}')
print('Files in adapter :', os.listdir(ADAPTER_PATH))

# Expose path to the server module via an env-var
os.environ['NUST_BANK_ADAPTER_PATH'] = ADAPTER_PATH

In [ ]:
from peft import PeftModel
from transformers import AutoTokenizer
import time

print(f'Applying LoRA adapter from {ADAPTER_PATH} ...')
t1 = time.time()

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'✅ LoRA weights applied in {time.time() - t1:.1f}s')
print(f'   Tokenizer vocab size: {tokenizer.vocab_size}')

# ── Inject the already-loaded objects into the server module so it
#    doesn't try to reload them from disk when the first request arrives.
import sys, importlib
sys.path.insert(0, '/content/LLM_Project')

import remote_llm_server as _srv
_srv._MODEL     = model
_srv._TOKENIZER = tokenizer
_srv.BASE_MODEL_ID = BASE_MODEL_ID
_srv.ADAPTER_PATH  = ADAPTER_PATH
_srv._DEVICE_INFO  = device_label

print('\n✅ Model injected into server module — server will NOT re-download on first request.')

## 6) Optional: Set API Token (Recommended)
Set a shared secret here. Use the same value locally as `NUST_BANK_REMOTE_LLM_TOKEN`.

In [ ]:
import os
os.environ['NUST_BANK_REMOTE_LLM_TOKEN'] = 'change_me_shared_secret'  # ← change this
print('Token set.')

## 7) Start Remote Model API Server

The server runs in a **background thread** so all log output appears directly in this cell.
Because the model was pre-loaded above, the server is ready to answer requests immediately.

In [ ]:
import threading
import remote_llm_server as _srv

SERVER_PORT = 8000

server_thread = threading.Thread(
    target=_srv.main,
    kwargs={'host': '0.0.0.0', 'port': SERVER_PORT},
    daemon=True,   # dies automatically when the kernel stops
)
server_thread.start()

import time, requests, sys

print(f'Waiting for server to become healthy on port {SERVER_PORT} ...')
sys.stdout.flush()

headers = {'Authorization': f"Bearer {os.environ.get('NUST_BANK_REMOTE_LLM_TOKEN', '')}"}

for attempt in range(1, 13):
    time.sleep(5)
    try:
        resp = requests.get(f'http://127.0.0.1:{SERVER_PORT}/health', headers=headers, timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            print(f'\n✅ Server healthy!')
            print(f'   Status      : {data.get("status")}')
            print(f'   Base model  : {data.get("base_model")}')
            print(f'   Device info : {data.get("device_info")}')
            print(f'   Model loaded: {data.get("model_loaded")}')
            break
        else:
            print(f'  Attempt {attempt}: HTTP {resp.status_code}')
    except Exception as exc:
        print(f'  Attempt {attempt}: {exc}')
    sys.stdout.flush()
else:
    raise RuntimeError('Server did not become healthy after 60 s — check logs above.')

## 8) Open ngrok Tunnel
If your ngrok account requires auth, set your token first.

In [ ]:
from pyngrok import ngrok

# Uncomment and set your ngrok auth token (required for most accounts)
# ngrok.set_auth_token('YOUR_NGROK_AUTH_TOKEN')

tunnel = ngrok.connect(SERVER_PORT)
public_url = tunnel.public_url

print('='*60)
print('  NGROK PUBLIC URL:', public_url)
print('='*60)
print()
print('Copy the URL above and use it on your local machine (see cell 9).')

## 9) Health Check via ngrok

In [ ]:
import requests, os

headers = {'Authorization': f"Bearer {os.environ.get('NUST_BANK_REMOTE_LLM_TOKEN', '')}"}

resp = requests.get(public_url + '/health', headers=headers, timeout=30)
print('Ngrok health status:', resp.status_code)
import json
print(json.dumps(resp.json(), indent=2))

## 10) Local Machine Commands
Run these in a terminal on your Windows laptop **before** starting Streamlit:

### CMD
```bat
set NUST_BANK_REMOTE_LLM_URL=https://xxxx.ngrok-free.app
set NUST_BANK_REMOTE_LLM_TOKEN=change_me_shared_secret
streamlit run streamlit_app.py
```

### PowerShell
```powershell
$env:NUST_BANK_REMOTE_LLM_URL="https://xxxx.ngrok-free.app"
$env:NUST_BANK_REMOTE_LLM_TOKEN="change_me_shared_secret"
streamlit run streamlit_app.py
```